# Two Feet + Head IMU Stride Analysis

This notebook analyzes walking gait using synchronized foot and head IMUs.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/stride_estimation_imu/blob/main/notebooks/demo_colab.ipynb)

In [ ]:
from datetime import time as dt_time

# =============================================================================
# FILE PATHS - Update these to point to your .h5 files
# Default: Use included demo data (Oct 29 pilot recording with left/right/head)
# =============================================================================
LEFT_FILE = 'data/20251029-154305_LF_Pilot_Ch_Oct29.h5'
RIGHT_FILE = 'data/20251029-154310_RF_Pilot_Ch_Oct29.h5'
HEAD_FILE = 'data/20251029-154313_Head_Pilot_Ch_Oct29.h5'
HAND_FILE = HEAD_FILE  # No hand sensor in this recording, use head as placeholder

# =============================================================================
# WALKING BOUT TARGETS
# Each tuple: (name, target_time, search_window_seconds)
# The script will find walking bouts near these times
# =============================================================================
WALKING_BOUT_TARGETS = [
    ('track walk', dt_time(16, 34, 0), 240),  # 4:34 PM, search +/- 240s
]

# =============================================================================
# BOUT DETECTION PARAMETERS
# =============================================================================
MIN_QUIET_SECONDS = 1.5   # Minimum quiet period to count as walk boundary
MIN_WALK_SECONDS = 3.0    # Minimum walking duration to include
W_THRESHOLD = 30.0        # Max angular velocity (deg/s) for quiet detection
A_THRESHOLD = 1.0         # Max accel deviation from gravity (m/s^2) for quiet

In [ ]:
from datetime import time as dt_time

# =============================================================================
# FILE PATHS - Update these to point to your .h5 files
# =============================================================================
LEFT_FILE = 'left_foot.h5'   # Left foot IMU
RIGHT_FILE = 'right_foot.h5' # Right foot IMU  
HEAD_FILE = 'head.h5'        # Head IMU
HAND_FILE = 'head.h5'        # Hand IMU (or set same as HEAD_FILE if not available)

# =============================================================================
# WALKING BOUT TARGETS
# Each tuple: (name, target_time, search_window_seconds)
# The script will find walking bouts near these times
# =============================================================================
WALKING_BOUT_TARGETS = [
    ('walk 1', dt_time(15, 0, 0), 120),   # 3:00 PM, search +/- 120s
    # ('walk 2', dt_time(15, 10, 0), 120), # Add more bouts as needed
]

# =============================================================================
# BOUT DETECTION PARAMETERS
# =============================================================================
MIN_QUIET_SECONDS = 1.5   # Minimum quiet period to count as walk boundary
MIN_WALK_SECONDS = 3.0    # Minimum walking duration to include
W_THRESHOLD = 30.0        # Max angular velocity (deg/s) for quiet detection
A_THRESHOLD = 1.0         # Max accel deviation from gravity (m/s^2) for quiet

## Setup

In [ ]:
# Clone repository and install dependencies (only needed in Colab)
!git clone https://github.com/jeremydwong/stride_estimation_imu.git 2>/dev/null || true
%cd stride_estimation_imu
!pip install -q numpy scipy matplotlib h5py

In [ ]:
import sys
import os
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

import stride_imu as imu
from stride_imu.apdm import load_imu_recording, find_overlapping_recordings, ImuRecording
from stride_imu.inertial import detect_walking_bouts, find_bouts_near_time, WalkingBout

print("stride_imu loaded successfully!")

## Upload Data Files (Colab only)

Run this cell to upload your .h5 files, then update the file paths in the Configuration cell above.

In [ ]:
# Upload files (Colab only)
try:
    from google.colab import files
    print("Select your APDM .h5 files to upload:")
    uploaded = files.upload()
    print(f"\nUploaded files: {list(uploaded.keys())}")
    print("\nUpdate the file paths in the Configuration cell above to match these filenames.")
except ImportError:
    print("Not running in Colab - using local file paths from Configuration cell.")

## Helper Functions

In [ ]:
class BoutSelection:
    """Stores bout selection with precise timing info."""
    def __init__(self, name, bout, start_datetime, end_datetime, confirmed=False):
        self.name = name
        self.bout = bout
        self.start_datetime = start_datetime
        self.end_datetime = end_datetime
        self.start_idx = bout.start_idx
        self.end_idx = bout.end_idx
        self.duration_seconds = bout.duration_seconds
        self.confirmed = confirmed

    def __repr__(self):
        return (f"BoutSelection('{self.name}', "
                f"start={self.start_datetime.strftime('%H:%M:%S')}, "
                f"duration={self.duration_seconds:.1f}s)")


def compute_alignment_rotation(P):
    """Compute rotation matrix to align trajectory with +Y axis."""
    direction = P[-1, :2] - P[0, :2]
    theta = np.arctan2(direction[1], direction[0])
    rotation_angle = np.pi / 2 - theta
    cos_a, sin_a = np.cos(rotation_angle), np.sin(rotation_angle)
    return np.array([[cos_a, -sin_a, 0], [sin_a, cos_a, 0], [0, 0, 1]])


def apply_rotation_and_offset(P, R, offset):
    """Apply rotation matrix and offset to trajectory."""
    P_centered = P - P[0, :]
    P_rotated = (R @ P_centered.T).T
    return P_rotated + offset


def compute_total_distance(P):
    """Compute total path length."""
    diffs = np.diff(P, axis=0)
    return float(np.sum(np.sqrt(np.sum(diffs ** 2, axis=1))))


def compute_step_metrics(strides):
    """Compute step-level metrics from stride segmentation."""
    step_speeds = strides['frwd_speed']
    step_durations = strides['time']
    
    if len(step_speeds) == 0:
        return {'step_lengths': np.array([]), 'step_durations': np.array([]),
                'step_speeds': np.array([]), 'mean_speed': 0.0, 'std_speed': 0.0,
                'mean_length': 0.0, 'std_length': 0.0, 'mean_duration': 0.0,
                'std_duration': 0.0, 'n_steps': 0}
    
    step_lengths = strides['frwd'][-1, :]
    return {
        'step_lengths': step_lengths, 'step_durations': step_durations,
        'step_speeds': step_speeds,
        'mean_speed': np.mean(step_speeds), 'std_speed': np.std(step_speeds),
        'mean_length': np.mean(step_lengths), 'std_length': np.std(step_lengths),
        'mean_duration': np.mean(step_durations), 'std_duration': np.std(step_durations),
        'n_steps': len(step_speeds)
    }


def analyze_head_motion(head_recording, period):
    """Analyze head IMU motion characteristics."""
    Wm = np.sqrt(np.sum(head_recording.Wb ** 2, axis=1)) / period * 180 / np.pi
    Am = np.sqrt(np.sum(head_recording.Ab ** 2, axis=1))
    return {
        'ang_vel_mean': np.mean(Wm), 'ang_vel_std': np.std(Wm), 'ang_vel_max': np.max(Wm),
        'accel_mean': np.mean(Am), 'accel_std': np.std(Am), 'accel_max': np.max(Am),
        'Wm': Wm, 'Am': Am
    }


def process_walking_bout(bout, left_rec, right_rec, head_rec, period):
    """Process a single walking bout."""
    left_bout = left_rec[bout.start_idx:bout.end_idx]
    right_bout = right_rec[bout.start_idx:bout.end_idx]
    head_bout = head_rec[bout.start_idx:bout.end_idx]
    
    left_walk_info, right_walk_info = imu.compute_position_two_imus(
        left_bout.Wb, left_bout.Ab, right_bout.Wb, right_bout.Ab, period)
    
    left_strides = imu.stride_segmentation(left_walk_info, period)
    right_strides = imu.stride_segmentation(right_walk_info, period)
    
    return {
        'left_metrics': compute_step_metrics(left_strides),
        'right_metrics': compute_step_metrics(right_strides),
        'head_metrics': analyze_head_motion(head_bout, period),
        'left_strides': left_strides, 'right_strides': right_strides,
        'left_walk_info': left_walk_info, 'right_walk_info': right_walk_info,
        'left_bout': left_bout, 'right_bout': right_bout, 'head_bout': head_bout,
        'bout': bout,
        'left_total_distance': compute_total_distance(left_walk_info['P']),
        'right_total_distance': compute_total_distance(right_walk_info['P'])
    }

## Plotting Functions

In [ ]:
def plot_rotation_corrected_trajectories(left_walk_info, right_walk_info, title="", dx_offset=0.3):
    """Plot left and right foot trajectories aligned to forward direction."""
    P_left, P_right = left_walk_info['P'], right_walk_info['P']
    R_left, R_right = compute_alignment_rotation(P_left), compute_alignment_rotation(P_right)
    P_left_rot = apply_rotation_and_offset(P_left, R_left, np.array([-dx_offset/2, 0, 0]))
    P_right_rot = apply_rotation_and_offset(P_right, R_right, np.array([dx_offset/2, 0, 0]))
    
    fig = plt.figure(figsize=(14, 6))
    fig.suptitle(f'Rotation Corrected Trajectories - {title}', fontsize=12)
    
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.plot(P_left_rot[:, 0], P_left_rot[:, 1], 'b-', linewidth=1, label='Left Foot', alpha=0.8)
    ax1.plot(P_right_rot[:, 0], P_right_rot[:, 1], 'r-', linewidth=1, label='Right Foot', alpha=0.8)
    ax1.plot(P_left_rot[0, 0], P_left_rot[0, 1], 'bo', markersize=8)
    ax1.plot(P_right_rot[0, 0], P_right_rot[0, 1], 'ro', markersize=8)
    ax1.set_xlabel('X (lateral) [m]'); ax1.set_ylabel('Y (forward) [m]')
    ax1.set_title('XY View'); ax1.legend(); ax1.grid(True, alpha=0.3); ax1.axis('equal')
    
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax2.plot(P_left_rot[:, 0], P_left_rot[:, 1], P_left_rot[:, 2], 'b-', linewidth=1, alpha=0.8)
    ax2.plot(P_right_rot[:, 0], P_right_rot[:, 1], P_right_rot[:, 2], 'r-', linewidth=1, alpha=0.8)
    ax2.set_xlabel('X [m]'); ax2.set_ylabel('Y [m]'); ax2.set_zlabel('Z [m]')
    ax2.set_title('3D View')
    plt.tight_layout()
    return fig


def plot_bout_summary(bout_name, left_metrics, right_metrics, head_metrics):
    """Create summary plot for a walking bout."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'Walking Bout: {bout_name}', fontsize=14)
    
    x = np.arange(2)
    
    # Step speed
    ax = axes[0, 0]
    means = [left_metrics['mean_speed'], right_metrics['mean_speed']]
    stds = [left_metrics['std_speed'], right_metrics['std_speed']]
    ax.bar(x, means, yerr=stds, capsize=5, color=['blue', 'red'], alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(['Left', 'Right'])
    ax.set_ylabel('Step Speed [m/s]'); ax.set_title('Mean Step Speed'); ax.grid(True, axis='y')
    
    # Step length
    ax = axes[0, 1]
    means = [left_metrics['mean_length'], right_metrics['mean_length']]
    stds = [left_metrics['std_length'], right_metrics['std_length']]
    ax.bar(x, means, yerr=stds, capsize=5, color=['blue', 'red'], alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(['Left', 'Right'])
    ax.set_ylabel('Step Length [m]'); ax.set_title('Mean Step Length'); ax.grid(True, axis='y')
    
    # Step speed distribution
    ax = axes[1, 0]
    all_speeds = []
    if left_metrics['n_steps'] > 0: all_speeds.extend(left_metrics['step_speeds'])
    if right_metrics['n_steps'] > 0: all_speeds.extend(right_metrics['step_speeds'])
    if all_speeds:
        bins = np.linspace(min(all_speeds), max(all_speeds), 46)
        if left_metrics['n_steps'] > 0:
            ax.hist(left_metrics['step_speeds'], bins=bins, alpha=0.5, label='Left', color='blue')
        if right_metrics['n_steps'] > 0:
            ax.hist(right_metrics['step_speeds'], bins=bins, alpha=0.5, label='Right', color='red')
    ax.set_xlabel('Step Speed [m/s]'); ax.set_ylabel('Count')
    ax.set_title('Step Speed Distribution'); ax.legend(); ax.grid(True)
    
    # Summary text
    ax = axes[1, 1]
    text = (f"Head Motion Summary:\n"
            f"Angular Velocity:\n  Mean: {head_metrics['ang_vel_mean']:.1f} deg/s\n"
            f"  Std: {head_metrics['ang_vel_std']:.1f} deg/s\n  Max: {head_metrics['ang_vel_max']:.1f} deg/s\n\n"
            f"Acceleration:\n  Mean: {head_metrics['accel_mean']:.2f} m/s²\n"
            f"  Std: {head_metrics['accel_std']:.2f} m/s²\n  Max: {head_metrics['accel_max']:.2f} m/s²\n\n"
            f"Steps: L={left_metrics['n_steps']}, R={right_metrics['n_steps']}")
    ax.text(0.1, 0.5, text, transform=ax.transAxes, fontsize=11, verticalalignment='center', fontfamily='monospace')
    ax.axis('off'); ax.set_title('Head Motion & Step Counts')
    
    plt.tight_layout()
    return fig

## 1. Load and Synchronize IMU Recordings

In [ ]:
print("Loading IMU recordings...")
left_rec = load_imu_recording(LEFT_FILE)
right_rec = load_imu_recording(RIGHT_FILE)
head_rec = load_imu_recording(HEAD_FILE)
hand_rec = load_imu_recording(HAND_FILE)

print(f"  Left foot: {len(left_rec)} samples")
print(f"  Right foot: {len(right_rec)} samples")
print(f"  Head: {len(head_rec)} samples")
print(f"  Hand: {len(hand_rec)} samples")

print("\nSynchronizing recordings...")
synced = find_overlapping_recordings([left_rec, right_rec, head_rec, hand_rec])
left_synced, right_synced, head_synced, hand_synced = synced
PERIOD = left_synced.period

print(f"Sampling period: {PERIOD:.6f} s ({1/PERIOD:.1f} Hz)")
print(f"Synchronized samples: {len(left_synced)}")

## 2. Detect Walking Bouts

In [ ]:
print("Detecting walking bouts...")
all_bouts = detect_walking_bouts(
    left_synced.Wb, left_synced.Ab, PERIOD,
    W_threshold=W_THRESHOLD, A_threshold=A_THRESHOLD,
    min_quiet_seconds=MIN_QUIET_SECONDS, min_walk_seconds=MIN_WALK_SECONDS
)

print(f"\nDetected {len(all_bouts)} walking bouts:")
for i, bout in enumerate(all_bouts):
    bout_start = left_synced.time_datetime[bout.start_idx]
    print(f"  {i+1}. {bout_start.strftime('%H:%M:%S')} - {bout.duration_seconds:.1f}s")

## 3. Select Bouts Near Target Times

In [ ]:
bout_selections = []

for bout_name, target_time, search_window in WALKING_BOUT_TARGETS:
    print(f"\nSearching for '{bout_name}' near {target_time.strftime('%H:%M:%S')}...")
    matching = find_bouts_near_time(all_bouts, left_synced.time_datetime, target_time, search_window)
    
    if matching:
        # Select longest matching bout
        best = max(matching, key=lambda b: b.duration_seconds)
        start_dt = left_synced.time_datetime[best.start_idx]
        end_dt = left_synced.time_datetime[best.end_idx - 1]
        sel = BoutSelection(bout_name, best, start_dt, end_dt, confirmed=True)
        bout_selections.append(sel)
        print(f"  Found: {sel}")
    else:
        print(f"  No bouts found within {search_window}s window")

print(f"\nSelected {len(bout_selections)} bout(s) for analysis")

## 4. Process Walking Bouts

In [ ]:
bout_results = {}

for sel in bout_selections:
    print(f"\n{'='*60}")
    print(f"Processing: {sel.name}")
    print(f"{'='*60}")
    
    result = process_walking_bout(sel.bout, left_synced, right_synced, head_synced, PERIOD)
    bout_results[sel.name] = {'selection': sel, **result}
    
    lm, rm, hm = result['left_metrics'], result['right_metrics'], result['head_metrics']
    print(f"Left:  {lm['n_steps']} steps, {lm['mean_speed']:.2f} m/s, {result['left_total_distance']:.2f} m")
    print(f"Right: {rm['n_steps']} steps, {rm['mean_speed']:.2f} m/s, {result['right_total_distance']:.2f} m")
    print(f"Head:  {hm['ang_vel_mean']:.1f} ± {hm['ang_vel_std']:.1f} deg/s")

## 5. Visualize Results

In [ ]:
for name, r in bout_results.items():
    lm, rm, hm = r['left_metrics'], r['right_metrics'], r['head_metrics']
    
    # Stride trajectories
    if lm['n_steps'] > 0 and rm['n_steps'] > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        plt.sca(axes[0])
        imu.plt_ltrl_frwd_strides(r['left_strides'], show=False)
        axes[0].set_title(f'Left Foot - {name}')
        plt.sca(axes[1])
        imu.plt_ltrl_frwd_strides(r['right_strides'], show=False)
        axes[1].set_title(f'Right Foot - {name}')
        plt.tight_layout()
        plt.show()
    
    # Rotation-corrected trajectories
    plot_rotation_corrected_trajectories(r['left_walk_info'], r['right_walk_info'], title=name)
    plt.show()
    
    # Bout summary
    plot_bout_summary(name, lm, rm, hm)
    plt.show()

## 6. Summary Table

In [ ]:
if bout_results:
    print(f"\n{'Bout':<15} {'Start':<10} {'Duration':>8} {'L Steps':>8} {'R Steps':>8} {'L Speed':>10} {'R Speed':>10} {'L Dist':>10} {'R Dist':>10}")
    print("-" * 100)
    for name, r in bout_results.items():
        sel = r['selection']
        lm, rm = r['left_metrics'], r['right_metrics']
        print(f"{name:<15} {sel.start_datetime.strftime('%H:%M:%S'):<10} "
              f"{sel.duration_seconds:>8.1f} {lm['n_steps']:>8} {rm['n_steps']:>8} "
              f"{lm['mean_speed']:>10.2f} {rm['mean_speed']:>10.2f} "
              f"{r['left_total_distance']:>10.2f} {r['right_total_distance']:>10.2f}")